# 20 — Are correct pairs still closer after endpoint and suffix controls?

## Question and prediction

Notebook 19 found greater text overlap and smaller `pre_last` distances among correct–correct pairs. This follow-up asks whether that distance contrast remains when compared under the same measured endpoint and suffix conditions.

**Prediction:** matching these text properties will reduce the positive II-minus-CC pair-distance gap relative to an unadjusted reference on the **same retained problems**. A residual gap is possible. This is an exploratory follow-up, not a preregistered independent confirmation.

The primary control uses actual last-token identities plus exact measured suffix overlap. The fixed summary is L24–L31; all layers are retained. This is **pair distance**, not Delta ID or correctness-probe accuracy.

## Three increasingly specific controls

1. `endpoint_equality`: compare CC and II pairs with the same status: their two endpoint tokens either match each other or differ. This is a coarse diagnostic; “equal” could mean two copies of different token identities in different pairs.
2. `endpoint_identity`: match the actual unordered pair of last-token IDs. A CC pair ending on [A,B] is compared with II pairs ending on [A,B] or [B,A].
3. **`endpoint_identity_suffix` (primary):** additionally match the number of identical trailing tokens (up to eight) and the compared window length. This exactly matches the prior suffix-overlap score. It does not match the complete wording of those windows, nor their reasoning content.

Every comparison is within the same problem. Empty generated prefixes/unusable endpoints have no control weight. These conditions use only text and class membership, never geometric distances, to define overlap.

## Exact overlap weighting instead of arbitrary pair subsampling

For each text group within a problem, count its CC and II pairs. If both occur, let its overlap mass be the smaller count. Spread that mass equally across its CC pairs and separately across its II pairs. Normalize so CC and II each have total weight one within the problem. Finally give every retained problem equal weight.

Example: a shared group with 2 CC and 6 II pairs has mass 2. Each CC pair initially gets weight 1 and each II pair weight 1/3. Both classes then give that group the same total weight. Normalization preserves the matched group distribution.

Unshared text groups receive zero weight. The notebook verifies exact joint balance. Problems without shared groups are explicitly excluded for that control; it does not relax the rules after seeing support or results. Each adjusted result is compared with all original pairs from those same retained problems. The full 88-problem reference remains separate.

This is a descriptive comparison on text overlap support. It is not a causal intervention or a demonstration that text mediates a fixed percentage of the ID effect. Pairs share attempts; no independent-pair confidence intervals or p-values are reported. Effective pair counts describe weight concentration, not independent sample sizes.

## Inputs and Colab steps

This notebook reuses **FINAL 19's exact distance checkpoints**. No GPU, Phi-2 model, new extraction, or new distance computation is needed. It reconstructs endpoint tokens using the tokenizer and frozen text, checking the original token hashes and Notebook 19 features.

1. Upload this standalone notebook to Colab. Keep QUICK=True and run numbered cells 1–7.
2. SOURCE19 normally resolves automatically to the original FINAL 19 run folder. If necessary, set it explicitly. It must contain `checkpoints/L00.npz` through `L31.npz`, not only an extracted compact review bundle.
3. Inspect/send the QUICK review bundle. QUICK computes all matching support and feature balance, but only three geometric layers (L0, L24, L31).
4. After review, set QUICK=False and rerun cells 1–7. FINAL reads all 32 source checkpoints and calculates the L24–L31 summary.

The original Notebook 19 and its expensive inputs do not need rerunning. If a source checkpoint is missing, first locate the original Drive run directory. Standard Colab dependencies: numpy, pandas, transformers, matplotlib, tqdm, pyarrow. Tokenizer serialization hashes are recorded for provenance. Every source prefix token sequence, generated suffix boundary, and endpoint/suffix feature must still match; actual token changes stop the run.

## How to read the results

- Positive II-minus-CC distance: correct pairs are closer.
- Negative adjusted-minus-reference gap shift: the gap becomes smaller after weighting on the same problem set.
- Remaining positive gap: the measured endpoint/suffix features do not fully account for the contrast on retained support. It does not prove shared reasoning.
- Zero/reversed gap: the original contrast is sensitive to this restriction. It does not establish absence of correctness-related information.
- Sparse/no overlap: limited/no ability to answer the primary question under these rules, not a null result.

Global lexical overlap, length and earlier value mentions are reported as residual feature balance and may remain different. Distinct controls can retain distinct problems; differences between their aggregate results are not automatically incremental effects of adding a feature.

In [ ]:
# 1 — Settings. CPU is sufficient; reuse FINAL 19's checkpoints, not its compact ZIP alone.
import json,hashlib,platform,re,zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
import transformers
from tqdm.auto import tqdm
from IPython.display import display
try:
    from google.colab import drive
except ImportError:pass
else:drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/msc_thesis')
SOURCE19=None # Original FINAL 19 run folder containing checkpoints/L00.npz ... L31.npz.
QUICK=True
LAYERS=[0,24,31] if QUICK else list(range(32))
EXPECTED19='970fd2d0ad90fc9973e1c796afc1fd0c5b964aa3800dd463aa2ad4bb6f4386d0'
FROZEN=BASE/'data/frozen/phi2_dataset_frozen_v1.parquet'
PRIMARY='endpoint_identity_suffix'
CONFIG=dict(version='endpoint_control_v1',quick=QUICK,layers=LAYERS,primary_control=PRIMARY,
    controls=['endpoint_equality','endpoint_identity','endpoint_identity_suffix'],
    suffix_window=8,weight='min class stratum counts; normalize within problem/class; equal problems',
    late_window=list(range(24,32)),source19_manifest_sha256=EXPECTED19,implementation_sha256='ca1f1fd1b57e77bb844c52459f263924c7ed81d36b8135886d1001c22f8872fe')
def digest(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()
print('QUICK' if QUICK else 'FINAL','| No generation, extraction, new pair distances or ID estimation')


In [ ]:
# 2 — Matching functions. No geometry is used to construct weights.
"""Exact within-problem text-stratum overlap weights; no distance-based selection."""
import numpy as np
import pandas as pd

CONTROLS={
 'endpoint_equality':['last_token_equal'],
 'endpoint_identity':['end_token_lo','end_token_hi'],
 'endpoint_identity_suffix':['end_token_lo','end_token_hi','suffix_count','suffix_denominator']}


def endpoint_features(a,b):
    if not a or not b:
        return dict(end_token_lo=-1,end_token_hi=-1,last_token_equal=np.nan,
                    suffix_count=-1,suffix_denominator=0,endpoint_available=False)
    lo,hi=sorted([int(a[-1]),int(b[-1])]);den=min(8,len(a),len(b));count=0
    for x,y in zip(reversed(a[-8:]),reversed(b[-8:])):
        if x!=y:break
        count+=1
    return dict(end_token_lo=lo,end_token_hi=hi,last_token_equal=int(a[-1]==b[-1]),
                suffix_count=count,suffix_denominator=den,endpoint_available=True)


def build_weights(pairs,columns):
    # Pair order is unchanged. Each class gets equal stratum weights inside a problem.
    valid=pairs.endpoint_available.to_numpy(bool)
    weights=np.zeros(len(pairs),np.float64);strata=[]
    for key,g in pairs[valid].groupby(['problem_id']+columns,sort=True,dropna=False):
        c=g.index[g.pair_type=='CC'].to_numpy();i=g.index[g.pair_type=='II'].to_numpy()
        mass=min(len(c),len(i))
        entry=dict(zip(['problem_id']+columns,key if isinstance(key,tuple) else (key,)))
        strata.append(dict(entry,n_CC=len(c),n_II=len(i),overlap_mass=mass))
        if mass:
            weights[c]=mass/len(c);weights[i]=mass/len(i)
    # Normalize independently within each problem/class. Totals are one in both classes.
    for (_,kind),g in pairs.groupby(['problem_id','pair_type']):
        total=weights[g.index].sum()
        if total:weights[g.index]/=total
    eligible=[]
    for pid,g in pairs.groupby('problem_id'):
        sums=[weights[g.index[g.pair_type==kind]].sum() for kind in ['CC','II']]
        if np.allclose(sums,[1,1]):eligible.append(int(pid))
        elif not np.allclose(sums,[0,0]):raise AssertionError('Unpaired problem support')
    return weights,np.asarray(eligible,int),pd.DataFrame(strata,columns=['problem_id']+columns+['n_CC','n_II','overlap_mass'])


def weighted_problem_values(pairs,values,weights,eligible):
    values=np.asarray(values,float);rows=[]
    if not np.isfinite(values).all():raise FloatingPointError('Invalid pair values')
    for pid in eligible:
        row=dict(problem_id=int(pid))
        for kind in ['CC','II']:
            ix=pairs.index[(pairs.problem_id==pid)&(pairs.pair_type==kind)].to_numpy()
            w=weights[ix];assert np.isclose(w.sum(),1)
            row[kind]=float(np.dot(w,values[ix]))
        row['II_minus_CC']=row['II']-row['CC'];rows.append(row)
    return pd.DataFrame(rows,columns=['problem_id','CC','II','II_minus_CC'])


def baseline_weights(pairs,eligible):
    w=np.zeros(len(pairs),float)
    for (_,kind),g in pairs[pairs.problem_id.isin(eligible)].groupby(['problem_id','pair_type']):w[g.index]=1/len(g)
    return w


def summarize_problem_values(df):
    if not len(df):return dict(n_problems=0,CC_mean=np.nan,II_mean=np.nan,gap_mean=np.nan,
        gap_median=np.nan,fraction_CC_closer=np.nan)
    return dict(n_problems=len(df),CC_mean=float(df.CC.mean()),II_mean=float(df.II.mean()),
        gap_mean=float(df.II_minus_CC.mean()),gap_median=float(df.II_minus_CC.median()),
        fraction_CC_closer=float((df.II_minus_CC>0).mean()))


In [ ]:
# 3 — Reconstruct endpoint identities independently of distances and validate against FINAL 19.
if SOURCE19 is None:
    candidates=list((BASE/'results/prefix_similarity_phi2_v1/final').glob('*/manifest.json'))
    matches=[p.parent for p in candidates if digest(p)==EXPECTED19]
    if len(matches)!=1:raise ValueError('Set SOURCE19 to the original FINAL 19 run directory containing checkpoints.')
    SOURCE19=matches[0]
SOURCE19=Path(SOURCE19)
assert digest(SOURCE19/'manifest.json')==EXPECTED19
m19=json.loads((SOURCE19/'manifest.json').read_text());assert m19['config']['quick'] is False
SOURCE_SIGNATURE=hashlib.sha256(json.dumps(m19,sort_keys=True).encode()).hexdigest()
assert digest(SOURCE19/'pair_features.csv')==m19['pair_features_sha256']
PAIRS=pd.read_csv(SOURCE19/'pair_features.csv').reset_index(drop=True)
ATTEMPTS=pd.read_csv(SOURCE19/'attempt_features.csv')
assert len(PAIRS)==11140 and len(ATTEMPTS)==2541
assert not PAIRS.duplicated(['a','b']).any() and (PAIRS.a<PAIRS.b).all()
assert np.array_equal(ATTEMPTS.row_index.to_numpy(),np.arange(len(ATTEMPTS)))
# Locate exactly the source 13 membership referenced by FINAL 19.
member_path=next(Path(k) for k in m19['source_files'] if Path(k).name=='cohort_membership.csv')
if not member_path.exists():
    candidates=list((BASE/'results/aligned_four_views_phi2_v1/final').glob('*/cohort_membership.csv'))
    matches=[p for p in candidates if digest(p)==m19['source_files'][str(member_path)]]
    if len(matches)!=1:raise ValueError('Cannot find source cohort_membership.csv in the FINAL 13 runs')
    member_path=matches[0]
assert digest(member_path)==next(v for k,v in m19['source_files'].items() if Path(k).name==member_path.name)
meta=pd.read_csv(member_path);excluded=meta.excluded.astype(str).str.lower().isin(['true','1'])
meta=meta[~excluded].reset_index(drop=True)
assert np.array_equal(meta[['problem_id','attempt_id']],ATTEMPTS[['problem_id','attempt_id']])
assert np.array_equal(meta.correct,ATTEMPTS.correct)
expected=next(v for k,v in m19['inputs'].items() if Path(k).name==FROZEN.name)
assert digest(FROZEN)==expected
frozen=pd.read_parquet(FROZEN).set_index(['problem_id','attempt_id']);assert frozen.index.is_unique
revision=m19.get('tokenizer_commit')
tok=AutoTokenizer.from_pretrained('microsoft/phi-2',use_fast=True,**({'revision':revision} if revision else {}));assert tok.is_fast
# Serialized backend state includes mutable truncation/padding settings.
# Notebook 19 recorded it AFTER encoding; a freshly loaded tokenizer may differ.
TOKENIZER_HASH_LOADED=hashlib.sha256(tok.backend_tokenizer.to_str().encode()).hexdigest()
if TOKENIZER_HASH_LOADED!=m19['tokenizer_backend_sha256']:
    print('Tokenizer serialization differs at load; checking all source prefix token IDs and endpoint features before proceeding.')
TOKENS=[];token_audit=[]
for j,row in tqdm(ATTEMPTS.iterrows(),total=len(ATTEMPTS),desc='Verify endpoint token identities'):
    f=frozen.loc[(int(row.problem_id),int(row.attempt_id))]
    text=str(f.generated_text);prompt=str(f.prompt);assert text.startswith(prompt)
    enc=tok(text,add_special_tokens=False,truncation=True,max_length=512,return_offsets_mapping=True)
    ids=enc['input_ids'];offsets=enc['offset_mapping'];cut=int(meta.prefix_T.iloc[j])
    assert len(ids)==int(meta.T_full.iloc[j]) and 0<cut<len(ids)
    h=hashlib.sha256(np.asarray(ids[:cut],np.int64).tobytes()).hexdigest();assert h==row.prefix_token_hash, f'Actual prefix token IDs changed for problem {row.problem_id}, attempt {row.attempt_id}; STOP.'
    positions=[t for t,(a,b) in enumerate(offsets[:cut]) if a>=len(prompt) and b>a]
    generated=[ids[t] for t in positions]
    assert len(generated)==int(row.generated_prefix_tokens), f'Generated prefix boundary changed at row {j}'
    # Exact full-prefix IDs plus this suffix boundary fix every generated token ID.
    assert positions==list(range(cut-len(generated),cut)), f'Noncontiguous generated prefix at row {j}'
    crossing=sum(a<len(prompt)<b for a,b in offsets[:cut])
    assert crossing==int(row.prompt_boundary_crossing_tokens), f'Prompt boundary changed at row {j}'
    TOKENS.append(generated)
    # Ensure the last generated token really is the vector's endpoint, not an earlier token.
    endpoint_ok=bool(generated) and offsets[cut-1][0]>=len(prompt) and offsets[cut-1][1]>offsets[cut-1][0]
    token_audit.append(dict(row_index=j,endpoint_available=endpoint_ok,
        end_token_id=int(ids[cut-1]),prefix_hash_verified=True))
TOKENIZER_HASH_AFTER_AUDIT=hashlib.sha256(tok.backend_tokenizer.to_str().encode()).hexdigest()
TOKEN_AUDIT=pd.DataFrame(token_audit)
extra=[]
for _,row in PAIRS.iterrows():
    a,b=int(row.a),int(row.b)
    assert ATTEMPTS.problem_id.iloc[a]==ATTEMPTS.problem_id.iloc[b]==row.problem_id
    expected_class=row.pair_type=='CC'
    assert bool(ATTEMPTS.correct.iloc[a])==bool(ATTEMPTS.correct.iloc[b])==expected_class
    features=endpoint_features(TOKENS[a],TOKENS[b])
    features['endpoint_available']=bool(features['endpoint_available'] and TOKEN_AUDIT.endpoint_available.iloc[a] and TOKEN_AUDIT.endpoint_available.iloc[b])
    if features['endpoint_available']:
        assert np.isclose(features['last_token_equal'],row.last_token_equal)
        assert np.isclose(features['suffix_count']/features['suffix_denominator'],row.last8_common_suffix_fraction)
    extra.append(features)
EXTRA=pd.DataFrame(extra)
PAIRS=PAIRS.drop(columns=['last_token_equal']).join(EXTRA)
assert PAIRS.problem_id.nunique()==88
print('Verified prefix hashes:',len(TOKEN_AUDIT),'| usable endpoint pairs:',int(PAIRS.endpoint_available.sum()),'/',len(PAIRS))


In [ ]:
# 4 — Freeze exact-stratum overlap weights using text and class membership only, never distances.
WEIGHTS={};ELIGIBLE={};BASELINES={};STRATA={};retention=[];BALANCE=[]
FEATURES=['last_token_equal','last8_common_suffix_fraction','token_bigram_jaccard',
    'masked_word_bigram_jaccard','generated_tokens_mean','abs_log_length_difference',
    'both_prior_selected_value','either_prior_selected_value']
for control,cols in CONTROLS.items():
    w,pids,strata=build_weights(PAIRS,cols)
    WEIGHTS[control]=w;ELIGIBLE[control]=pids;STRATA[control]=strata
    BASELINES[control]=baseline_weights(PAIRS,pids)
    for pid,g in PAIRS.groupby('problem_id'):
        s=strata[strata.problem_id==pid]
        row=dict(control=control,problem_id=int(pid),retained=bool(pid in pids),
            shared_strata=int((s.overlap_mass>0).sum()),overlap_mass=float(s.overlap_mass.sum()))
        for kind in ['CC','II']:
            ix=g.index[g.pair_type==kind].to_numpy();row[kind+'_pairs_before']=len(ix)
            row[kind+'_pairs_positive_weight']=int((w[ix]>0).sum())
            row[kind+'_effective_pair_count']=float(1/np.sum(w[ix]**2)) if (w[ix]>0).any() else 0
        retention.append(row)
    # Exact balance of the FULL controlled joint stratum, not just one marginal mean.
    for key,g in PAIRS[PAIRS.problem_id.isin(pids)].groupby(['problem_id']+cols):
        c=w[g.index[g.pair_type=='CC']].sum();i=w[g.index[g.pair_type=='II']].sum()
        assert np.isclose(c,i,atol=1e-12),f'Unbalanced stratum: {control}, {key}'
    for feature in FEATURES:
        values=PAIRS[feature].to_numpy(float)
        if not np.isfinite(values).all():raise ValueError(f'Missing feature {feature}; review rather than silently fill')
        for condition,weights in [('reference_same_problems',BASELINES[control]),('adjusted',w)]:
            f=weighted_problem_values(PAIRS,values,weights,pids)
            BALANCE.append(dict(control=control,condition=condition,feature=feature,n_problems=len(f),
                CC_mean=f.CC.mean(),II_mean=f.II.mean(),CC_minus_II=(f.CC-f.II).mean()))
RETENTION=pd.DataFrame(retention);BALANCE=pd.DataFrame(BALANCE)
SUMMARY_RETENTION=RETENTION.groupby('control',as_index=False).agg(problems_before=('problem_id','size'),
    retained_problems=('retained','sum'),shared_strata=('shared_strata','sum'),overlap_mass=('overlap_mass','sum'),
    CC_pairs_positive_weight=('CC_pairs_positive_weight','sum'),II_pairs_positive_weight=('II_pairs_positive_weight','sum'))
MANIFEST=dict(config=CONFIG,source19=str(SOURCE19),source19_signature=SOURCE_SIGNATURE,
    source_pair_sha256=m19['pair_features_sha256'],source19_manifest_sha256=EXPECTED19,
    weights_sha256={k:hashlib.sha256(v.tobytes()).hexdigest() for k,v in WEIGHTS.items()},
    source_tokenizer_backend_sha256=m19['tokenizer_backend_sha256'],
    tokenizer_backend_sha256=TOKENIZER_HASH_AFTER_AUDIT,
    tokenizer_backend_sha256_at_load=TOKENIZER_HASH_LOADED,
    tokenizer_validation='exact source prefix hashes, generated suffix boundaries, and pair endpoint/suffix features',
    versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,transformers=transformers.__version__))
SIGNATURE=hashlib.sha256(json.dumps(MANIFEST,sort_keys=True).encode()).hexdigest()
OUT=BASE/'results/endpoint_suffix_control_phi2_v1'/('quick' if QUICK else 'final')/SIGNATURE[:12]
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'manifest.json').write_text(json.dumps(MANIFEST,indent=2))
for name,df in [('pair_features',PAIRS),('endpoint_audit',TOKEN_AUDIT),('retention_by_problem',RETENTION),('retention_summary',SUMMARY_RETENTION),('feature_balance',BALANCE)]:df.to_csv(OUT/(name+'.csv'),index=False)
for control in CONTROLS:
    STRATA[control].to_csv(OUT/(control+'_strata.csv'),index=False)
pd.DataFrame(dict(a=PAIRS.a,b=PAIRS.b,**WEIGHTS)).to_csv(OUT/'pair_weights.csv',index=False)
print('Results:',OUT);display(SUMMARY_RETENTION)
print('Primary control:',PRIMARY,'| retained problems:',len(ELIGIBLE[PRIMARY]))
print('Unsupported strata receive zero weight. No automatic relaxation if overlap is sparse.')
display(BALANCE.query('condition=="adjusted"').round(4))


In [ ]:
# 5 — Read original FINAL 19 distance checkpoints; verify every used layer against its source summary.
SOURCE_GEOM=pd.read_csv(SOURCE19/'geometry_by_problem.csv')
DISTANCES={};CHECKS=[]
for layer in LAYERS:
    path=SOURCE19/'checkpoints'/f'L{layer:02d}.npz'
    if not path.exists():raise RuntimeError(f'Missing {path}. Use the original FINAL 19 run directory, not the compact review ZIP.')
    with np.load(path,allow_pickle=False) as z:
        assert str(z['signature'].item())==SOURCE_SIGNATURE
        assert np.array_equal(z['a'],PAIRS.a) and np.array_equal(z['b'],PAIRS.b)
        d=z['distance'].copy()
    assert len(d)==len(PAIRS) and np.isfinite(d).all() and (d>1e-12).all()
    full=weighted_problem_values(PAIRS,d,baseline_weights(PAIRS,PAIRS.problem_id.unique()),PAIRS.problem_id.unique()).set_index('problem_id')
    ref=SOURCE_GEOM[SOURCE_GEOM.window==f'L{layer:02d}'].set_index('problem_id').loc[full.index]
    for kind in ['CC','II']:
        error=float(np.abs(full[kind]-ref[kind]).max());passed=bool(np.allclose(full[kind],ref[kind],rtol=1e-10,atol=1e-12))
        CHECKS.append(dict(layer=layer,pair_type=kind,max_abs_difference=error,passed=passed))
        if not passed:raise RuntimeError('Source geometry reproduction failed')
    DISTANCES[f'L{layer:02d}']=d
HAVE_LATE=set(range(24,32)).issubset(LAYERS)
if HAVE_LATE:DISTANCES['L24-L31']=np.mean([DISTANCES[f'L{l:02d}'] for l in range(24,32)],axis=0)
else:print('QUICK: no L24–L31 aggregate.')
CHECKS=pd.DataFrame(CHECKS);CHECKS.to_csv(OUT/'source_reproduction_checks.csv',index=False)
print('Verified source layer/class checks:',len(CHECKS))


In [ ]:
# 6 — Paired retained-problem comparisons. No pair-level p-values or independent-pair CIs.
rows=[];problem_rows=[];contrast_rows=[];full_rows=[]
for window,d in DISTANCES.items():
    full=weighted_problem_values(PAIRS,d,baseline_weights(PAIRS,PAIRS.problem_id.unique()),PAIRS.problem_id.unique())
    full_rows.append(dict(window=window,**summarize_problem_values(full)))
    for control in CONTROLS:
        if not len(ELIGIBLE[control]):continue
        frames={}
        for condition,w in [('reference_same_problems',BASELINES[control]),('adjusted',WEIGHTS[control])]:
            f=weighted_problem_values(PAIRS,d,w,ELIGIBLE[control]);frames[condition]=f
            rows.append(dict(window=window,control=control,condition=condition,**summarize_problem_values(f)))
            problem_rows.append(f.assign(window=window,control=control,condition=condition))
        a=frames['adjusted'].set_index('problem_id');b=frames['reference_same_problems'].set_index('problem_id')
        shift=a[['CC','II','II_minus_CC']]-b[['CC','II','II_minus_CC']]
        for pid,s in shift.iterrows():contrast_rows.append(dict(window=window,control=control,problem_id=int(pid),
            C_shift=s.CC,I_shift=s.II,gap_shift=s.II_minus_CC))
RESULTS=pd.DataFrame(rows);FULL=pd.DataFrame(full_rows)
PROBLEM=pd.concat(problem_rows,ignore_index=True) if problem_rows else pd.DataFrame()
CONTRAST=pd.DataFrame(contrast_rows)
CONTRAST_SUMMARY=(CONTRAST.groupby(['window','control'],as_index=False).agg(n_problems=('problem_id','size'),
    C_shift_mean=('C_shift','mean'),I_shift_mean=('I_shift','mean'),gap_shift_mean=('gap_shift','mean'),
    gap_shift_median=('gap_shift','median')) if len(CONTRAST) else pd.DataFrame())
for name,df in [('distance_summary',RESULTS),('distance_by_problem',PROBLEM),('paired_adjustment_by_problem',CONTRAST),('paired_adjustment_summary',CONTRAST_SUMMARY),('full_88_problem_reference',FULL)]:df.to_csv(OUT/(name+'.csv'),index=False)
window='L24-L31' if HAVE_LATE else 'L31'
if len(RESULTS):display(RESULTS[RESULTS.window==window].round(4))
else:print('No shared support in any control. This is a support limitation; do not relax rules automatically.')
if not len(ELIGIBLE[PRIMARY]):print('PRIMARY QUESTION NOT ESTIMABLE: no shared endpoint/suffix strata.')
print('Positive gap = incorrect pairs farther apart. Compare adjusted with its SAME-PROBLEM reference, not only with all 88 problems.')


In [ ]:
# 7 — Figures and review bundle.
FIG=OUT/'figures';FIG.mkdir(exist_ok=True)
if len(RESULTS):
    for control in CONTROLS:
        g=RESULTS[(RESULTS.control==control)&(RESULTS.window!='L24-L31')].copy()
        if not len(g):continue
        g['layer']=g.window.str[1:].astype(int)
        fig,axes=plt.subplots(1,2,figsize=(12,4))
        for condition in ['reference_same_problems','adjusted']:
            h=g[g.condition==condition].sort_values('layer')
            axes[0].plot(h.layer,h.CC_mean,label=condition+' CC');axes[0].plot(h.layer,h.II_mean,ls='--',label=condition+' II')
            axes[1].plot(h.layer,h.gap_mean,label=condition)
        axes[0].set(xlabel='Layer',ylabel='Mean pair distance');axes[0].legend(fontsize=7)
        axes[1].axhline(0,color='gray',lw=.8);axes[1].set(xlabel='Layer',ylabel='II minus CC distance');axes[1].legend(fontsize=7)
        fig.suptitle(f'{control} | {len(ELIGIBLE[control])} shared problems | '+('QUICK selected layers' if QUICK else 'All layers'))
        fig.tight_layout();fig.savefig(FIG/(control+'.png'),dpi=150);plt.show();plt.close(fig)
notes='''# Endpoint and suffix controls

Read manifest for QUICK/FINAL. Reuses FINAL 19 pair distances exactly; no new ID estimation, extraction or model generation. Source 19 contains 88 common pair problems, with 1056 CC and 10084 II pairs. Source distance summaries are reproduced for every used layer.

Three controls: endpoint_equality matches only whether the pair's last tokens match each other. endpoint_identity matches the actual UNORDERED token-ID pair. endpoint_identity_suffix (primary) additionally matches the number of identical final tokens up to eight and the compared suffix-window length; this exactly balances the prior suffix score. It does not match the full wording of all eight tokens. Pairs without usable generated endpoints have zero control weight.

Matching is within problem. For each shared text stratum, mass=min(n_CC,n_II). Every CC pair in it gets mass/n_CC and every II pair mass/n_II, then weights are normalized within problem/class. All retained problems receive equal weight. No distances inform strata or weights. The procedure balances empirical joint endpoint/suffix distributions exactly on their overlap without arbitrary pair subsampling. Effective pair counts are weight-concentration diagnostics, not independent sample sizes.

Each adjusted comparison has a reference using ALL original CC and II pairs from the SAME retained problems with equal problem weights. Full-88 reference is separate. Controls may retain different problems and cannot be compared as if the populations were identical. No automatic relaxation of sparse overlap. Zero shared support is reported as non-estimable, not a null effect.

Feature balance reports unadjusted/adjusted means on retained problems. Endpoint identities and the full controlled joint strata are checked directly for exact balance. Global lexical overlap, length, earlier numeric mentions and reasoning content are not necessarily balanced. The primary control answers a restricted association question, not a causal explanation or mediation analysis.

Positive II-minus-CC distance means correct pairs are closer. A smaller adjusted gap supports an association of measured text properties with part of that contrast. A remaining gap does not prove shared reasoning. A zero or reversed gap does not establish that correctness-related information is absent. No inferential p-values or independent-pair confidence intervals: pairs share attempts and this cohort informed the design.

The actual token IDs of the last prefix token are reconstructed from the exact verified source prefix hashes. Shared prompt text is excluded from suffix features. The source inherited numeric answer cut, duplicate filtering, stored precision and exploratory cohort limitations remain.

QUICK uses all text pairs and all matching strata but only L0/L24/L31 distances. FINAL uses all 32 layers and L24–L31. Thus support and feature balance are already complete in QUICK. Save the original FINAL 19 folder with checkpoints; its compact review bundle alone is insufficient to rerun this notebook.
'''
(OUT/'REVIEW_NOTES.md').write_text(notes)
with zipfile.ZipFile(OUT/'review_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.glob('*.csv'))+[OUT/'manifest.json',OUT/'REVIEW_NOTES.md']+sorted(FIG.glob('*.png')):z.write(path,path.relative_to(OUT))
print('Review bundle:',OUT/'review_bundle.zip')
